# Visualização do Grafo de Fluxo Urbano
**Chicago Taxi Trips — E5**

Visualiza os hexágonos H3 no mapa real de Chicago, coloridos por:
- Volume de fluxo total
- Comunidade detectada
- PageRank

In [ ]:
import pandas as pd
import folium
import h3
import json
from pathlib import Path

# Carrega os resultados da E5
metrics     = pd.read_csv('outputs/graph/node_metrics.csv')
communities = pd.read_csv('outputs/graph/communities.csv')

# Junta métricas com comunidades
df = metrics.merge(communities, on='h3_cell', how='left')

print(f'Hexágonos: {len(df)}')
print(f'Comunidades: {df["community"].nunique()}')
print(f'\nTop 5 por fluxo:')
print(df[['h3_cell', 'total_flow', 'pagerank', 'community']].head())

In [ ]:
# ── Mapa 1: Volume de fluxo total ─────────────────────────

def h3_to_polygon(h3_cell):
    """Converte célula H3 para polígono GeoJSON."""
    boundary = h3.cell_to_boundary(h3_cell)
    # h3 retorna (lat, lon) — folium quer [lat, lon]
    coords = [[lat, lon] for lat, lon in boundary]
    return coords

# Normaliza fluxo para opacidade (0.2 a 0.9)
df['flow_norm'] = (df['total_flow'] - df['total_flow'].min()) / \
                  (df['total_flow'].max() - df['total_flow'].min())
df['opacity']   = df['flow_norm'] * 0.7 + 0.2

# Cores por intensidade de fluxo
def flow_color(norm):
    if norm > 0.8:   return '#800026'
    elif norm > 0.6: return '#BD0026'
    elif norm > 0.4: return '#E31A1C'
    elif norm > 0.2: return '#FC4E2A'
    elif norm > 0.1: return '#FD8D3C'
    elif norm > 0.05: return '#FEB24C'
    else:            return '#FFEDA0'

# Cria mapa centrado em Chicago
m1 = folium.Map(location=[41.85, -87.65], zoom_start=11, tiles='CartoDB positron')

for _, row in df.iterrows():
    try:
        coords  = h3_to_polygon(row['h3_cell'])
        color   = flow_color(row['flow_norm'])
        tooltip = (
            f"H3: {row['h3_cell']}<br>"
            f"Fluxo total: {int(row['total_flow']):,}<br>"
            f"Entrada: {int(row['in_strength']):,}<br>"
            f"Saída: {int(row['out_strength']):,}<br>"
            f"PageRank: {row['pagerank']:.6f}<br>"
            f"Comunidade: {int(row['community'])}"
        )
        folium.Polygon(
            locations=coords,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=row['opacity'],
            weight=0.5,
            tooltip=tooltip
        ).add_to(m1)
    except Exception:
        pass

m1.save('outputs/graph/map_flow.html')
print('Mapa salvo em outputs/graph/map_flow.html')
m1

In [ ]:
# ── Mapa 2: Comunidades ────────────────────────────────────

COMMUNITY_COLORS = [
    '#E63946', '#457B9D', '#2A9D8F', '#E9C46A', '#F4A261',
    '#264653', '#A8DADC', '#F1FAEE', '#6A4C93', '#1982C4'
]

m2 = folium.Map(location=[41.85, -87.65], zoom_start=11, tiles='CartoDB positron')

for _, row in df.iterrows():
    try:
        coords    = h3_to_polygon(row['h3_cell'])
        comm_id   = int(row['community'])
        color     = COMMUNITY_COLORS[comm_id % len(COMMUNITY_COLORS)]
        tooltip   = (
            f"H3: {row['h3_cell']}<br>"
            f"Comunidade: {comm_id}<br>"
            f"Fluxo total: {int(row['total_flow']):,}"
        )
        folium.Polygon(
            locations=coords,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.6,
            weight=0.5,
            tooltip=tooltip
        ).add_to(m2)
    except Exception:
        pass

m2.save('outputs/graph/map_communities.html')
print('Mapa salvo em outputs/graph/map_communities.html')
m2

In [ ]:
# ── Mapa 3: PageRank ───────────────────────────────────────

df['pr_norm'] = (df['pagerank'] - df['pagerank'].min()) / \
                (df['pagerank'].max() - df['pagerank'].min())

def pagerank_color(norm):
    if norm > 0.8:    return '#003566'
    elif norm > 0.6:  return '#0077B6'
    elif norm > 0.4:  return '#0096C7'
    elif norm > 0.2:  return '#00B4D8'
    elif norm > 0.05: return '#90E0EF'
    else:             return '#CAF0F8'

m3 = folium.Map(location=[41.85, -87.65], zoom_start=11, tiles='CartoDB positron')

for _, row in df.iterrows():
    try:
        coords  = h3_to_polygon(row['h3_cell'])
        color   = pagerank_color(row['pr_norm'])
        tooltip = (
            f"H3: {row['h3_cell']}<br>"
            f"PageRank: {row['pagerank']:.6f}<br>"
            f"Fluxo total: {int(row['total_flow']):,}"
        )
        folium.Polygon(
            locations=coords,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.7,
            weight=0.5,
            tooltip=tooltip
        ).add_to(m3)
    except Exception:
        pass

m3.save('outputs/graph/map_pagerank.html')
print('Mapa salvo em outputs/graph/map_pagerank.html')
m3

In [ ]:
# ── Análise temporal ───────────────────────────────────────
import plotly.express as px

with open('outputs/graph/temporal_analysis.json') as f:
    temporal = json.load(f)

# Fluxo por hora
df_hourly = pd.DataFrame(temporal['hourly_flow'])
fig = px.bar(
    df_hourly, x='hour', y='trips',
    title='Volume de corridas por hora do dia',
    labels={'hour': 'Hora', 'trips': 'Corridas'},
    color='trips', color_continuous_scale='Reds'
)
fig.show()

# Fluxo por estação
df_season = pd.DataFrame(temporal['seasonal_flow'])
fig2 = px.bar(
    df_season, x='season', y='trips',
    title='Volume de corridas por estação',
    labels={'season': 'Estação', 'trips': 'Corridas'},
    color='trips', color_continuous_scale='Blues'
)
fig2.show()

In [ ]:
# ── Resumo estatístico ─────────────────────────────────────
print('=== TOP 10 HEXÁGONOS POR FLUXO TOTAL ===')
print(df[['rank', 'h3_cell', 'total_flow', 'in_strength', 'out_strength', 
          'pagerank', 'community']].head(10).to_string(index=False))

print('\n=== DISTRIBUIÇÃO POR COMUNIDADE ===')
print(df.groupby('community')['total_flow'].agg(['count', 'sum', 'mean']).round(0))